# H1 模型侧校准诊断（M5 · 分析与统稿）

**回答的问题**：除开市场（B6）的校准斜率审计（M2 出），我们的模型（B0–B5b）自己在哪些概率区间、哪个结果类别上偏离校准？方向与市场是相同还是相反？

**方法**：对每个 arm 的每一类做 one-vs-rest 校准：
- 可靠性曲线：横轴=预测概率，纵轴=实际频率，用**高斯核局部平滑**（Nadaraya–Watson，W7），不用硬分箱；
- 校准斜率/截距：`logit(实际) ~ a + b·logit(预测)`，b<1=over-dispersed（过度自信），b>1=under-dispersed（太保守）；
- ECE 与分区域偏差（低尾 <8%、高尾 >85%、平局 20–35%）。

数据：`results/preds/preds_B*.csv`（每行含 `p_home/p_draw/p_away/y`，无需重建 panel）。

In [ ]:
import sys, numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
%matplotlib inline

ROOT = Path.cwd()
PREDS = ROOT / "results" / "preds"
OUT = ROOT / "results" / "logs"
FIG = ROOT / "products" / "figures"
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

ARMS = ["B0", "B1", "B2", "B3", "B4", "B5a", "B5b", "B6"]   # B6 作为市场参照；B6r ≈ B6
CLS = [("home", 0, "p_home"), ("draw", 1, "p_draw"), ("away", 2, "p_away")]
GRID = np.linspace(0.01, 0.99, 99)
BW = 0.045   # 概率尺度上的高斯核带宽（局部平滑）

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def kernel_smooth(x, z, grid, bw):
    """Nadaraya-Watson 局部平滑；z in {0,1}"""
    out = np.full_like(grid, np.nan, dtype=float)
    for i, g in enumerate(grid):
        w = np.exp(-0.5 * ((x - g) / bw) ** 2)
        sw = w.sum()
        if sw > 1e-9:
            out[i] = float(w @ z) / sw
    return out

def logistic_slope(x, z):
    """one-vs-rest logistic: z ~ a + b*x (Newton-Raphson). 返回 intercept, slope, se_slope"""
    X = np.column_stack([np.ones_like(x), x])
    b = np.zeros(2)
    for _ in range(100):
        eta = X @ b
        mu = 1 / (1 + np.exp(-np.clip(eta, -30, 30)))
        g = X.T @ (z - mu)
        W = mu * (1 - mu) + 1e-10
        H = X.T @ (W[:, None] * X)
        step = np.linalg.solve(H, g)
        b += step
        if np.abs(step).max() < 1e-9:
            break
    eta = X @ b
    mu = 1 / (1 + np.exp(-np.clip(eta, -30, 30)))
    W = mu * (1 - mu) + 1e-10
    se = np.sqrt(np.diag(np.linalg.inv(X.T @ (W[:, None] * X))))
    return b[0], b[1], se[1]

def ece_binned(p, z, nb=10):
    edges = np.linspace(0, 1, nb + 1); n = len(p); tot = 0.0
    for i in range(nb):
        m = (p >= edges[i]) & (p < edges[i + 1]) if i < nb - 1 else (p >= edges[i]) & (p <= edges[i + 1])
        if m.sum() == 0:
            continue
        tot += (m.sum() / n) * abs(p[m].mean() - z[m].mean())
    return tot

In [ ]:
rel_rows, slope_rows, ece_rows, bias_rows = [], [], [], []
for arm in ARMS:
    df = pd.read_csv(PREDS / f"preds_{arm}.csv")
    y = df["y"].to_numpy(dtype=int)
    for name, c, col in CLS:
        p = df[col].to_numpy(dtype=float)
        z = (y == c).astype(float)
        sm = kernel_smooth(p, z, GRID, BW)
        for xv, yv in zip(GRID, sm):
            rel_rows.append({"arm": arm, "cls": name, "x": float(xv), "y": float(yv)})
        a, s, se_s = logistic_slope(logit(p), z)
        slope_rows.append({"arm": arm, "cls": name, "intercept": float(a), "slope": float(s), "se_slope": float(se_s)})
        ece_rows.append({"arm": arm, "cls": name, "ece": float(ece_binned(p, z))})
        for lo, hi, tag in [(0.0, 0.08, "low_tail"), (0.85, 1.0, "high_tail")]:
            m = (p >= lo) & (p <= hi)
            if m.sum() >= 20:
                bias_rows.append({"arm": arm, "cls": name, "region": tag,
                                  "n": int(m.sum()), "bias": float(p[m].mean() - z[m].mean())})
        if name == "draw":
            m = (p >= 0.20) & (p <= 0.35)
            if m.sum() >= 20:
                bias_rows.append({"arm": arm, "cls": name, "region": "draw_mid",
                                  "n": int(m.sum()), "bias": float(p[m].mean() - z[m].mean())})

rel_df = pd.DataFrame(rel_rows)
slope_df = pd.DataFrame(slope_rows)
ece_df = pd.DataFrame(ece_rows)
bias_df = pd.DataFrame(bias_rows)

rel_df.to_csv(OUT / "m5_reliability_curves.csv", index=False)
slope_df.to_csv(OUT / "m5_calibration_slopes.csv", index=False)
ece_df.to_csv(OUT / "m5_ece.csv", index=False)
bias_df.to_csv(OUT / "m5_regional_bias.csv", index=False)

print("校准斜率（行=arm, 列=类）：")
print(slope_df.pivot(index="arm", columns="cls", values="slope").round(3))
print("\n平局 20–35% 区间 bias（pred − freq，正=高估）：")
print(bias_df[(bias_df.cls == "draw") & (bias_df.region == "draw_mid")].set_index("arm")["bias"].round(4))

**一句话结论**：模型侧与市场侧方向相反——B4 三个类别的校准斜率都 < 1（over-dispersed / 过度自信，平局最低 0.78），而市场 B6 斜率 > 1（under-dispersed / FLB）。换更强模型（B5a/B5b）在平局上的校准反而更差（斜率 0.32–0.38，ECE ≈ 0.04）。

In [ ]:
ARM_COLOR = {"B0": "#9e9e9e", "B1": "#b0bec5", "B2": "#607d8b", "B3": "#4caf50",
             "B4": "#1f4e9c", "B5a": "#e67e22", "B5b": "#d62728", "B6": "#000000"}
CLS_LABEL = {"home": "Home win", "draw": "Draw", "away": "Away win"}
MODEL_ARMS = ["B0", "B1", "B2", "B3", "B4", "B5a", "B5b"]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, (cls, cid, col) in zip(axes, CLS):
    ax.plot([0, 1], [0, 1], color="#555", lw=1, ls="--", label="Perfect calibration")
    for arm in MODEL_ARMS:
        d = rel_df[(rel_df.arm == arm) & (rel_df.cls == cls)].dropna()
        ax.plot(d["x"], d["y"], color=ARM_COLOR[arm], lw=(2.4 if arm == "B4" else 1.1),
                label=(arm if arm in ("B4", "B0", "B5b") else "_nolegend_"))
    db6 = rel_df[(rel_df.arm == "B6") & (rel_df.cls == cls)].dropna()
    ax.plot(db6["x"], db6["y"], color="black", lw=1.4, ls=":", label="B6 (market)")
    ax.set_title(f"{CLS_LABEL[cls]}  ({col})"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Predicted probability")
    if cls == "home":
        ax.set_ylabel("Observed frequency")
axes[0].legend(loc="upper left", frameon=False, ncol=2, fontsize=7)
fig.suptitle("fig05 · Model-side reliability curves (local smoothing) — over-dispersed, opposite of market", fontsize=10)
fig.savefig(FIG / "fig05_model_reliability.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# fig06：市场侧（数字来自 M2 的 h1_market.ipynb，此处仅排版）
devig = ["proportional", "power", "shin"]
slope = [1.0864, 1.0305, 1.0515]; lo = [1.0472, 0.9938, 1.0144]; hi = [1.1244, 1.0659, 1.0882]
low = np.array([0.0124, 0.0067, 0.0016]); high = np.array([0.0018, -0.0103, -0.0221])

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6), gridspec_kw={"width_ratios": [1, 1.15]})
ax = axes[0]; y = np.arange(3)[::-1]
for yi, s, l, h in zip(y, slope, lo, hi):
    ax.errorbar(s, yi, xerr=[[s - l], [h - s]], fmt="o", color="#1f4e9c", capsize=3)
ax.axvline(1.0, color="#555", ls="--", lw=1)
ax.set_yticks(y); ax.set_yticklabels([d.capitalize() for d in devig][::-1])
ax.set_xlabel("Calibration slope (90% cluster CI)"); ax.set_title("Market: slope > 1 (under-dispersed)", fontsize=9.5)
ax.set_xlim(0.95, 1.16)
ax = axes[1]; x = np.arange(3); w = 0.38
ax.bar(x - w/2, low, w, color="#d62728", label="Low tail (<8%): overpriced longshots")
ax.bar(x + w/2, high, w, color="#1f77b4", label="High tail (>85%): underpriced favourites")
ax.axhline(0, color="#555", lw=1); ax.set_xticks(x); ax.set_xticklabels(["Home", "Draw", "Away"])
ax.set_ylabel("Tail-bin bias (pred − freq)"); ax.set_title("Favourite–longshot bias by class", fontsize=9.5)
ax.legend(frameon=False, fontsize=7)
fig.suptitle("fig06 · Market-side H1 (slope 1.086, 90% CI 1.047–1.124)", fontsize=10)
fig.savefig(FIG / "fig06_market_calibration.png", dpi=300, bbox_inches="tight")
plt.show()

## 结论（供报告 §5.2 使用）

1. **市场 under-dispersed**：B6 校准斜率 1.086（90% CI 1.047–1.124）> 1，低概率区被高估、高概率区被低估（FLB）——H1 主判据支持。
2. **模型 over-dispersed**：B4 斜率 0.92/0.78/0.94，方向与市场相反；平局最差。
3. **更强的模型没修复校准**：B5a/B5b 在平局上斜率仅 0.32–0.38、ECE ≈ 0.04，是阶梯里平局校准最差的。

这三条拼起来就是完整的 H1：**市场与模型各有方向相反、形状不同的校准缺陷**，且「换更强的模型」没有让模型更校准。